**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->

# Dalle feature all’apprendimento
**Computer Vision 2026 · Lezione 9 · lunedì 12 ottobre · Concepts, 2 ore**

Il laboratorio esteso di apprendimento si svolgerà nella lezione successiva del 15 ottobre.
Questo quaderno usa solo CPU e immagini locali: nessun download durante l’esperimento e nessuna rete neurale da addestrare.

Al termine sapremo distinguere rappresentazione, parametri e iperparametri;
separare training, validation e test; leggere errori e metriche; riconoscere una valutazione contaminata.

Eseguire dall’alto in basso con **Restart Kernel and Run All**. Le celle di codice sono complete;
le domande richiedono interpretazione. I risultati vengono salvati nella cartella `risultati_lezione09`.


Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

<!-- C02 -->

## 1. Dalle immagini alle righe di una matrice

Il file locale contiene 1797 immagini 8×8 delle cifre 0–9, con valori interi 0–16,
esportate da `sklearn.datasets.load_digits` 1.6.1. È il sottoinsieme di test del dataset UCI originale:
qui lo ripartizioniamo per un **nuovo esperimento didattico**, senza rivendicare il protocollo originale UCI.
Autori: E. Alpaydin e C. Kaynak (1998), licenza CC BY 4.0. Provenienza e trasformazioni in `dati/README.md`.

La rappresentazione scelta è $\phi(I)=\operatorname{vec}(I)\in\mathbb{R}^{64}$.
Appiattire conserva i valori ma non impone al classificatore alcuna conoscenza della vicinanza tra pixel.
Non stiamo apprendendo le feature: apprendiamo una regola sulle intensità fissate.

[Documentazione Digits](https://scikit-learn.org/1.6/modules/generated/sklearn.datasets.load_digits.html) ·
[Fonte UCI](https://doi.org/10.24432/C50P49).

In [ ]:
# C03 — cella 3 del notebook
from pathlib import Path
import os, sys, json, hashlib, platform, csv
import importlib.metadata as metadata

# Cache e output restano nella cartella di lavoro; nessun percorso personale richiesto.
os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.mplconfig'))
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '2')
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay, log_loss)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.facecolor': 'white'})
BASE = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dati/digits_cv2026.npz').exists()), None)
if BASE is None:
    raise FileNotFoundError('Estrarre tutto lo ZIP e aprire il notebook nella cartella estratta.')
RESULTS = Path.cwd() / 'risultati_lezione09'
RESULTS.mkdir(exist_ok=True)
DATA_FILE = BASE / 'dati/digits_cv2026.npz'
data = np.load(DATA_FILE, allow_pickle=False)
images = data['images']
y = data['target'].astype(int)
X = images.reshape(len(images), -1).astype(np.float64)
assert X.shape == (1797, 64) and set(y) == set(range(10))
assert X.min() == 0 and X.max() == 16 and np.isfinite(X).all()
versions = {n: metadata.version(n) for n in
            ['numpy', 'scipy', 'scikit-learn', 'matplotlib', 'nbformat', 'nbclient', 'ipykernel']}
print('Immagini:', images.shape, '| Matrice X:', X.shape, '| Python:', platform.python_version())
print('Librerie:', versions)

<!-- C04 -->

## 2. Il protocollo si decide prima dei risultati

Partizione stratificata per **immagine**, circa 60% training, 20% validation, 20% test.
Le etichette sono usate dal partizionatore per mantenere le proporzioni, non per scegliere il modello.
Gli indici fissati sono inclusi nel pacchetto e ricostruiti qui per verificarli.

1. **Train:** stima della media e della deviazione standard, memorizzazione dei vicini, apprendimento di $W, b$.
2. **Validation:** scelta di $k$, distanza e regolarizzazione; selezione del modello principale.
3. **Test:** valutazione finale dei due modelli già scelti. Nessuna modifica dopo averlo osservato.

La stratificazione non crea indipendenza tra autori. Qui manca l’identità dello scrivente:
un autore potrebbe comparire in più partizioni. Non possiamo concludere che il modello generalizzi
a scriventi mai visti, fotografie, altre risoluzioni o altri domini.

In [ ]:
# C05 — cella 5 del notebook
SEED_TEST, SEED_VALIDATION = 20261012, 20261013
idx = np.arange(len(y))
trainval_idx, test_idx = train_test_split(
    idx, test_size=0.20, random_state=SEED_TEST, stratify=y)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.25, random_state=SEED_VALIDATION,
    stratify=y[trainval_idx])
splits = {'train': train_idx, 'validation': val_idx, 'test': test_idx}
assert np.array_equal(train_idx, data['train_idx'])
assert np.array_equal(val_idx, data['val_idx'])
assert np.array_equal(test_idx, data['test_idx'])
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert np.array_equal(np.sort(np.concatenate(list(splits.values()))), idx)
# Anche i contenuti delle 1797 immagini sono distinti; non basta però a separare gli autori.
assert len(np.unique(X, axis=0)) == len(X)
print('Partizioni disgiunte:', {name: len(ids) for name, ids in splits.items()})
print('Il test resta sigillato fino alla sezione 7.')

# In questa fase osserviamo immagini ed etichette soltanto del training.
fig, axes = plt.subplots(2, 10, figsize=(13, 3.8), constrained_layout=True)
for digit in range(10):
    ids = train_idx[y[train_idx] == digit][:2]
    for row, i in enumerate(ids):
        axes[row, digit].imshow(images[i], cmap='gray_r', vmin=0, vmax=16,
                                interpolation='nearest')
        axes[row, digit].set_title(f'y={y[i]} · id {i}', fontsize=9)
        axes[row, digit].axis('off')
fig.suptitle('Digits: due esempi di training per classe · immagini 8×8')
fig.savefig(RESULTS / '01_digits_training.png', dpi=180)
plt.show()

<!-- C06 -->

### Domanda 1

Se disponessimo di più fotografie della stessa persona, basterebbe uno split casuale per immagine? Quale informazione aggiuntiva servirebbe?

**La nostra risposta:** …

<!-- C07 -->

## 3. Due famiglie di modelli, una stessa rappresentazione

**kNN:** trova i $k$ vettori di training più vicini e vota. Proviamo distanza L1 ($p=1$)
e L2 ($p=2$), pesi uniformi. Non ottimizza $W, b$: conserva gli esempi. Anche qui esiste
una fase `fit`, e il costo di previsione dipende dal numero di esempi conservati.

**Classificatore lineare softmax:** $s(x)=Wz+b$, dove $z=(x-\mu_{train})/\sigma_{train}$.
Dieci score e una decisione $\hat y=\arg\max_c s_c$. Usiamo regressione logistica multiclasse,
cross-entropy e penalità L2. Usiamo $C$ per il **numero di classi** (qui $C=10$) e
$\lambda$ per il coefficiente della penalità nell’obiettivo delle figure. Chiamiamo
**`C_reg`** l’iperparametro di scikit-learn inverso alla forza della regolarizzazione:
un valore piccolo regolarizza di più. L’API della libreria lo chiama `C`, quindi il codice
scrive `LogisticRegression(C=C_reg)`. `C_reg` non è il numero di classi né il coefficiente
$\lambda$; il collegamento con quest’ultimo dipende anche dalla convenzione usata per loss e penalità.

La standardizzazione fa parte della scelta metodologica fissata prima dell’esperimento.
Influenza distanze e regolarizzazione; non è sempre la migliore rappresentazione per le immagini.
La `Pipeline` stima lo scaler solo quando eseguiamo `fit(X_train, y_train)` e applica quelle
statistiche immutate a validation e test. Le feature costanti nel training sono gestite dallo scaler.

[API ufficiale del modello lineare](https://scikit-learn.org/1.6/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [ ]:
# C08 — cella 8 del notebook
# Griglie piccole, stabilite PRIMA di calcolare qualsiasi metrica.
K_VALUES, P_VALUES = [1, 3, 5, 9], [1, 2]
C_REG_VALUES = [0.01, 0.1, 1.0, 10.0]
models, validation_rows = {}, []

def metrics_from_predictions(truth, prediction):
    precision, recall, f1, _ = precision_recall_fscore_support(
        truth, prediction, labels=np.arange(10), average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(truth, prediction)),
            'precision_macro': float(precision), 'recall_macro': float(recall),
            'macro_f1': float(f1)}

def fit_and_validate(name, family, estimator, hyperparameters):
    pipe = make_pipeline(StandardScaler(), estimator)
    pipe.fit(X[train_idx], y[train_idx])
    # Controllo utile: nessuna statistica è stata stimata su validation/test.
    scaler = pipe.named_steps['standardscaler']
    assert scaler.n_samples_seen_ == len(train_idx)
    np.testing.assert_allclose(scaler.mean_, X[train_idx].mean(axis=0))
    train_scores = metrics_from_predictions(y[train_idx], pipe.predict(X[train_idx]))
    val_scores = metrics_from_predictions(y[val_idx], pipe.predict(X[val_idx]))
    row = {'name': name, 'family': family, **hyperparameters, **val_scores,
           'train_macro_f1': train_scores['macro_f1']}
    if family == 'Lineare':
        assert np.all(pipe[-1].n_iter_ < pipe[-1].max_iter), 'Ottimizzazione non convergente'
        row['validation_cross_entropy'] = float(log_loss(y[val_idx], pipe.predict_proba(X[val_idx])))
    models[name] = pipe
    validation_rows.append(row)

for p in P_VALUES:
    for k in K_VALUES:
        fit_and_validate(f'kNN k={k}, p={p}', 'kNN',
            KNeighborsClassifier(n_neighbors=k, p=p, weights='uniform', algorithm='brute'),
            {'k': k, 'p': p})
for C_reg in C_REG_VALUES:
    fit_and_validate(f'Lineare C_reg={C_reg:g}', 'Lineare',
        LogisticRegression(C=C_reg, solver='lbfgs', penalty='l2', max_iter=3000,
                           tol=1e-8, random_state=SEED_TEST), {'C_reg': C_reg})

print(f'{"Candidato":22s} {"F1 train":>10s} {"F1 val":>10s} {"Acc val":>10s}')
for r in validation_rows:
    print(f"{r['name']:22s} {r['train_macro_f1']:10.4f} {r['macro_f1']:10.4f} {r['accuracy']:10.4f}")

In [ ]:
# C09 — cella 9 del notebook
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for p, color in [(1, '#337da8'), (2, '#b4562e')]:
    rows = [r for r in validation_rows if r['family'] == 'kNN' and r['p'] == p]
    axes[0].plot(K_VALUES, [r['macro_f1'] for r in rows], 'o-', color=color,
                 label=f'Validation, L{p}')
axes[0].set(xlabel='k: numero di vicini', ylabel='Macro-F1', title='kNN · scelta di k e distanza',
            xticks=K_VALUES, ylim=(0.90, 1.005))
axes[0].legend()
linear_rows = [r for r in validation_rows if r['family'] == 'Lineare']
for key, label, style in [('train_macro_f1', 'Training', '--'), ('macro_f1', 'Validation', '-')]:
    axes[1].semilogx(C_REG_VALUES, [r[key] for r in linear_rows], 'o' + style, label=label)
axes[1].set(xlabel='C_reg: regolarizzazione più debole verso destra', ylabel='Macro-F1',
            title='Classificatore lineare · scelta di C_reg', ylim=(0.90, 1.005))
axes[1].legend()
fig.savefig(RESULTS / '02_validation.png', dpi=180)
plt.show()

<!-- C10 -->

### Domanda 2

k=1 ottiene un risultato quasi perfetto sul training: è una prova di buona generalizzazione? Indicare inoltre due parametri e due iperparametri del modello lineare.

**La nostra risposta:** …

<!-- C11 -->

## 4. Congelare le scelte

La metrica primaria è **macro-F1 di validation**, per dare a ogni classe lo stesso peso.
Scegliamo un candidato per famiglia, poi il modello principale fra i due vincitori.
In caso di parità esatta, teniamo il primo nell’ordine delle griglie dichiarate.
L’accuracy è riportata, ma non modifica la regola di selezione.

Il confronto finale dei due vincitori è previsto fin dall’inizio. Conserviamo le pipeline già
addestrate sui 1077 esempi: non facciamo refit su train+validation, per mantenere evidente
quali dati hanno stimato ogni quantità. Il test non viene ancora usato.

In [ ]:
# C12 — cella 12 del notebook
frozen_winners = {
    family: max((r for r in validation_rows if r['family'] == family), key=lambda r: r['macro_f1'])
    for family in ['kNN', 'Lineare']
}
principal = max(frozen_winners.values(), key=lambda r: r['macro_f1'])['name']
frozen_protocol = {
    'seed_test': SEED_TEST, 'seed_validation': SEED_VALIDATION,
    'split': {name: len(ids) for name, ids in splits.items()},
    'primary_selection_metric': 'validation macro_f1; exact ties: first grid candidate',
    'fit_data': 'train only, no refit',
    'data_sha256': hashlib.sha256(DATA_FILE.read_bytes()).hexdigest(),
    'frozen_winners': frozen_winners, 'principal_model': principal,
}
frozen_json = json.dumps(frozen_protocol, ensure_ascii=False, indent=2)
(RESULTS / 'protocollo_congelato.json').write_text(frozen_json, encoding='utf-8')
print('Vincitori fissati:', {family: r['name'] for family, r in frozen_winners.items()})
print('Modello principale scelto su validation:', principal)
print('Decisione salvata prima della prima previsione sul test.')

<!-- C13 -->

### Domanda 3

Perché media e deviazione standard vanno stimate sul training anche se non usano direttamente le etichette?

**La nostra risposta:** …

<!-- C14 -->

## 5. Vedere dentro $s=Wz+b$

Per un batch $Z\in\mathbb{R}^{N\times64}$, scikit-learn conserva $W\in\mathbb{R}^{10\times64}$
e calcola $S=ZW^T+b$. Le mappe dei pesi sotto sono espresse nelle coordinate **standardizzate**.
Un peso positivo aumenta lo score della propria classe, a parità delle altre feature;
la decisione dipende dal confronto fra score, non dal singolo peso isolato.

Nel titolo della figura, `C` è il nome dell’argomento della libreria. Il valore scelto è `C_reg=1`.

In [ ]:
# C15 — cella 15 del notebook
linear_model = models[frozen_winners['Lineare']['name']]
scaler, classifier = linear_model[0], linear_model[1]
Z_val = scaler.transform(X[val_idx])
W, b = classifier.coef_, classifier.intercept_
scores_val = Z_val @ W.T + b
np.testing.assert_allclose(scores_val, linear_model.decision_function(X[val_idx]), atol=1e-10)
np.testing.assert_array_equal(classifier.classes_[scores_val.argmax(axis=1)],
                              linear_model.predict(X[val_idx]))
print('Z:', Z_val.shape, '| W:', W.shape, '| b:', b.shape, '| score:', scores_val.shape)

fig, axes = plt.subplots(2, 5, figsize=(10, 4.6), constrained_layout=True)
bound = np.abs(W).max()
for cls, ax in enumerate(axes.flat):
    im = ax.imshow(W[cls].reshape(8, 8), cmap='RdBu_r', vmin=-bound, vmax=bound,
                   interpolation='nearest')
    ax.set_title(f'Classe {cls}'); ax.axis('off')
fig.colorbar(im, ax=axes.ravel().tolist(), label='Peso sul pixel standardizzato', shrink=.85)
fig.suptitle('W appreso sui soli esempi di training · modello lineare C scelto su validation')
fig.savefig(RESULTS / '03_pesi_lineari.png', dpi=180)
plt.show()

<!-- C16 -->

## 6. Score, probabilità e loss non sono la stessa cosa

La softmax stabile sottrae il massimo di ogni riga:
$$p_c=\frac{\exp(s_c-m)}{\sum_j\exp(s_j-m)}, \qquad m=\max_j s_j.$$
La sottrazione non cambia i rapporti e evita overflow. Non trasforma automaticamente
le probabilità in frequenze calibrate: la calibrazione è una proprietà da verificare su dati adatti.

La cross-entropy per un esempio è $-\log p_y$; qui usiamo logaritmi naturali (unità: nat).
L’accuracy guarda soltanto l’argmax. In addestramento la regressione logistica minimizza
la cross-entropy con una penalità L2; `log_loss` riporta invece la sola componente sui dati,
senza penalità. Macro-F1 è la metrica scelta per selezionare il candidato, non la loss ottimizzata.

In [ ]:
# C17 — cella 17 del notebook
def stable_softmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=-1, keepdims=True)

def cross_entropy_from_scores(scores, labels):
    # Log-sum-exp stabile: evita di prendere il log di una probabilità sottofluita a zero.
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=int)
    shifted = scores - scores.max(axis=1, keepdims=True)
    return -shifted[np.arange(len(labels)), labels] + np.log(np.exp(shifted).sum(axis=1))

extreme_scores = np.array([[1000., 1001., 999.]])
p_extreme = stable_softmax(extreme_scores)
assert np.isfinite(p_extreme).all()
np.testing.assert_allclose(p_extreme.sum(axis=1), 1.)
np.testing.assert_allclose(stable_softmax(extreme_scores - 1000), p_extreme)
np.testing.assert_allclose(stable_softmax(scores_val),
                           linear_model.predict_proba(X[val_idx]), atol=1e-10)
np.testing.assert_allclose(cross_entropy_from_scores(scores_val, y[val_idx]).mean(),
                           log_loss(y[val_idx], stable_softmax(scores_val)))
print('Softmax di [1000, 1001, 999]:', p_extreme.round(6))

# Un esempio costruito, separato dal dataset: stessa classe corretta, diversa sicurezza.
toy_scores = np.array([[2., 1., 0.], [10., 5., 0.]])
toy_labels = np.array([0, 0])
toy_p = stable_softmax(toy_scores)
toy_losses = cross_entropy_from_scores(toy_scores, toy_labels)
for name, probs, loss in zip(['A', 'B'], toy_p, toy_losses):
    print(f'{name}: p={probs.round(4)}, classe={probs.argmax()}, accuracy=1, loss={loss:.6f}')

fig, ax = plt.subplots(figsize=(7.2, 3.8), constrained_layout=True)
positions = np.arange(3)
for j, (name, color) in enumerate([('A', '#337da8'), ('B', '#b4562e')]):
    ax.bar(positions + (j - .5) * .32, toy_p[j], width=.32, color=color,
           label=f'{name}: stessa accuracy, CE={toy_losses[j]:.3f}')
ax.set(xticks=positions, xticklabels=['0 (vera)', '1', '2'], ylabel='Probabilità softmax',
       xlabel='Classe', ylim=(0, 1.18), title='L’accuracy resta 1; la cross-entropy cambia')
ax.legend(loc='upper right')
fig.savefig(RESULTS / '04_loss_metriche.png', dpi=180)
plt.show()

<!-- C18 -->

### Domanda 4

Moltiplicando tutti gli score per una costante positiva, la classe predetta cambia? E la loss? Un errore molto sicuro viene penalizzato poco o molto dalla cross-entropy?

**La nostra risposta:** …

<!-- C19 -->

## 7. Aprire il test una sola volta nel protocollo

Le scelte sono ora congelate. Calcoliamo le metriche sui due candidati già dichiarati.
Ripetere una stessa esecuzione per verificarne la riproducibilità è lecito; usare i risultati
per cambiare griglia, feature o vincitore consumerebbe il ruolo del test come valutazione finale.

Nella confusion matrix $M$, **righe = classe vera, colonne = classe predetta**.
Per la classe $c$: precisione $P_c=M_{cc}/\sum_i M_{ic}$, richiamo $R_c=M_{cc}/\sum_j M_{cj}$,
$F1_c=2P_cR_c/(P_c+R_c)$. Macro-F1 è la media dei dieci $F1_c$,
non l’F1 calcolato a partire dalle due medie di precisione e richiamo.
Qui i denominatori nulli sono convenzionalmente gestiti con zero.

In [ ]:
# C20 — cella 20 del notebook
# Prima e unica cella che produce predizioni sul TEST: i nomi vengono dal protocollo salvato.
assert (RESULTS / 'protocollo_congelato.json').read_text(encoding='utf-8') == frozen_json
test_rows, test_predictions, matrices = {}, {}, {}
for family, selected in frozen_winners.items():
    pipe = models[selected['name']]
    prediction = pipe.predict(X[test_idx])
    test_predictions[family] = prediction
    M = confusion_matrix(y[test_idx], prediction, labels=np.arange(10))
    matrices[family] = M
    row = {'model': selected['name'], **metrics_from_predictions(y[test_idx], prediction),
           'n_errors': int((prediction != y[test_idx]).sum()), 'confusion_matrix': M.tolist()}
    if family == 'Lineare':
        row['cross_entropy'] = float(log_loss(y[test_idx], pipe.predict_proba(X[test_idx])))
    test_rows[family] = row

print('Modello principale (decisione presa su validation):', principal)
print(f'{"Modello":22s} {"Accuracy":>9s} {"Macro P":>9s} {"Macro R":>9s} {"Macro F1":>9s} {"Errori":>8s}')
for row in test_rows.values():
    print(f"{row['model']:22s} {row['accuracy']:9.4f} {row['precision_macro']:9.4f} "
          f"{row['recall_macro']:9.4f} {row['macro_f1']:9.4f} {row['n_errors']:5d}/360")

fig, axes = plt.subplots(1, 2, figsize=(11.6, 5), constrained_layout=True)
for ax, family in zip(axes, ['kNN', 'Lineare']):
    ConfusionMatrixDisplay(matrices[family], display_labels=np.arange(10)).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set(title=f"{test_rows[family]['model']} · {test_rows[family]['n_errors']} errori / 360",
           xlabel='Classe predetta', ylabel='Classe vera')
fig.savefig(RESULTS / '05_confusion_matrix.png', dpi=180)
plt.show()

In [ ]:
# C21 — cella 21 del notebook
# Ricostruiamo precisione, richiamo e F1 a partire dai conteggi del kNN.
M = matrices['kNN']
tp = np.diag(M).astype(float)
predicted_count, true_count = M.sum(axis=0), M.sum(axis=1)
precision_class = np.divide(tp, predicted_count, out=np.zeros(10), where=predicted_count != 0)
recall_class = np.divide(tp, true_count, out=np.zeros(10), where=true_count != 0)
f1_class = np.divide(2 * precision_class * recall_class, precision_class + recall_class,
                     out=np.zeros(10), where=(precision_class + recall_class) != 0)
np.testing.assert_allclose(f1_class.mean(), test_rows['kNN']['macro_f1'])
np.testing.assert_allclose(tp.sum() / M.sum(), test_rows['kNN']['accuracy'])
print(f'{"Classe":>6s} {"Supporto":>8s} {"Precision":>10s} {"Recall":>10s} {"F1":>10s}')
for digit in range(10):
    print(f'{digit:6d} {true_count[digit]:8d} {precision_class[digit]:10.4f} '
          f'{recall_class[digit]:10.4f} {f1_class[digit]:10.4f}')

<!-- C22 -->

### Domanda 5

Per il kNN, la classe 8 ha 31 previsioni corrette su 35 casi veri. Qual è il richiamo? Per calcolare la precisione guardiamo la riga o la colonna?

**La nostra risposta:** …

<!-- C23 -->

## 8. Gli errori hanno un aspetto

Mostriamo tutti gli errori del modello principale nell’ordine degli indici del dataset,
senza scegliere a mano quelli più convincenti. Le etichette restano quelle del dataset:
la somiglianza visiva suggerisce ipotesi, non dimostra la causa dell’errore.

In [ ]:
# C24 — cella 24 del notebook
principal_family = next(f for f, r in frozen_winners.items() if r['name'] == principal)
pred = test_predictions[principal_family]
wrong_positions = np.flatnonzero(pred != y[test_idx])
wrong_positions = wrong_positions[np.argsort(test_idx[wrong_positions])]
cols = 5
rows = max(1, int(np.ceil(len(wrong_positions) / cols)))
fig, axes = plt.subplots(rows, cols, figsize=(10, 2.25 * rows), squeeze=False,
                         constrained_layout=True)
for ax in axes.flat:
    ax.axis('off')
error_records = []
for ax, pos in zip(axes.flat, wrong_positions):
    original_id = int(test_idx[pos])
    ax.imshow(images[original_id], cmap='gray_r', vmin=0, vmax=16, interpolation='nearest')
    ax.set_title(f'id {original_id}: vero {y[original_id]} → pred. {pred[pos]}', fontsize=10)
    error_records.append({'image_id': original_id, 'true': int(y[original_id]),
                          'predicted': int(pred[pos])})
fig.suptitle(f'Tutti gli errori del modello principale: {principal} · ordine per id')
fig.savefig(RESULTS / '06_errori_test.png', dpi=180)
plt.show()

report = {**frozen_protocol, 'versions': versions, 'python': platform.python_version(),
          'class_counts': {name: np.bincount(y[ids], minlength=10).tolist()
                           for name, ids in splits.items()},
          'validation': validation_rows, 'test': test_rows,
          'principal_test_errors': error_records,
          'softmax_demo': {'probabilities': toy_p.tolist(), 'cross_entropy': toy_losses.tolist()},
          'limitations': 'Image-level split; writer identities unavailable; no unseen-writer claim.'}
(RESULTS / 'metriche_esecuzione.json').write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
with (RESULTS / 'metriche_test.csv').open('w', newline='', encoding='utf-8') as f:
    fields = ['model', 'accuracy', 'precision_macro', 'recall_macro', 'macro_f1', 'n_errors']
    writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    writer.writeheader(); writer.writerows(test_rows.values())
with (RESULTS / 'errori_modello_principale.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['image_id', 'true', 'predicted'])
    writer.writeheader(); writer.writerows(error_records)
print('Figure, protocollo, metriche e tabella degli errori salvati in:', RESULTS)

<!-- C25 -->

### Domanda 6

Il modello lineare ottiene un errore in meno del kNN su questo test. Possiamo sostituire adesso il modello principale? Una differenza di un caso su 360 dimostra che una famiglia è superiore?

**La nostra risposta:** …

<!-- C26 -->

### Domanda 7

Indicare un errore osservato, una possibile spiegazione e una verifica futura coerente con il protocollo. Distinguere osservazione e ipotesi.

**La nostra risposta:** …

<!-- C27 -->

## Consegna facoltativa in autonomia di riflessione

Per consolidare la lezione, completare in autonomia il notebook con le sette risposte, la figura della confusion matrix e
`metriche_test.csv`. In massimo 150 parole, formulare una conclusione che specifichi:
dataset e unità di split; scelta fatta su validation; una metrica finale;
un errore concreto; una generalizzazione che questi dati non permettono di sostenere.
Non serve cercare un risultato migliore né ampliare la griglia.

### Collegamento alle sorgenti già adottate

- `DeepLearning_Book/classificazione.tex`: sezioni «Nearest Neighbor», «Classificatore lineari»,
  «Funzione Loss», «Regolarizzazione», «Classificatore Softmax».
- `old_slide_CV2025/14_image_classification_linear_class.pdf` e sorgente `.pptx`:
  classificazione come assegnazione di etichette, distanze, score lineari e loss.
- L’esempio Digits, le verifiche numeriche e le domande sono stati costruiti per la lezione 9.
  Non riusano i risultati storici di CIFAR-10 come misure di questo esperimento.

Le sorgenti teoriche rimangono quelle del corso; i collegamenti ufficiali alle API servono
solo a documentare dati e implementazione. Consultati il 12 settembre 2026.